# Лабораторная работа 4. Часть 1 — Гипотезы

**ФИО:** Иноземцев Сергей Викторович  
**Группа:** ЕТ-216

Исходные данные: `Пренатальная диагностика.xlsx`, лист `исходные данные`.
Цель: исследовать распределения показателей и сравнить рост, вес и ИМТ мальчиков и девочек.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
DATA = Path('.')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight')
    plt.show()
    plt.close('all')


## Постановка и выбор методов
Выполняются прикреплённые задания «Гипотезы», «Т-тесты» и бонус «Проверка нормальности». Сравниваются независимые группы, поэтому используется двусторонний t-тест Уэлча (equal_var=False), не требующий равенства дисперсий. H0: средние равны; H1: средние различаются; α=0,05. Для серии сравнений дополнительно приведена поправка Holm. Неотклонение H0 не доказывает равенство распределений. Номер учащегося и код профиля не считаются измеряемыми психологическими шкалами.

## Новорождённые Проверка структуры и единиц

In [ ]:
birth=pd.read_excel(DATA/'Пренатальная диагностика.xlsx',sheet_name='исходные данные')
display(birth.head());display(birth.isna().sum());display(birth['pol(1-м, 0-д)'].value_counts().rename(index={0:'девочки',1:'мальчики'}))
assert set(birth['pol(1-м, 0-д)'].unique())<={0,1}
birth['ИМТ']=(birth.ves/1000)/(birth.rost/100)**2
num=[c for c in birth.select_dtypes('number').columns if c!='pol(1-м, 0-д)']
descr=birth[num].describe().T;descr['range']=birth[num].max()-birth[num].min();descr['median']=birth[num].median();display(descr)

ИМТ рассчитан как масса в килограммах / рост в метрах²; исходные единицы — граммы и сантиметры. Здесь показатель используется только как учебный числовой признак, без медицинских заключений. Код пола бинарный и не проверяется на непрерывное нормальное распределение.

In [ ]:
normal=[]
fig,axes=plt.subplots(3,3,figsize=(15,12))
for col,ax in zip(num,axes.flat):
    x=birth[col].dropna();sns.histplot(x,kde=True,ax=ax);ax.set_title(col)
    test=stats.shapiro(x);normal.append([col,len(x),x.mean(),x.median(),stats.skew(x,bias=False),stats.kurtosis(x,bias=False),test.statistic,test.pvalue])
for ax in axes.flat[len(num):]:ax.axis('off')
show('birth_normality')
normal=pd.DataFrame(normal,columns=['показатель','n','mean','median','skew','excess','W','p']);normal['H0 нормальность отклонена']=normal.p<.05;display(normal)

При нормальном распределении теоретические асимметрия и эксцесс равны 0, среднее и медиана совпадают. Выборочные оценки не обязаны быть точно нулевыми. Shapiro проверяет H0 о нормальности; при большом n он выявляет и небольшие отклонения. Визуализация и величина отклонений нужны вместе с p-value.

In [ ]:
def holm(pvalues):
    p=np.asarray(pvalues);order=np.argsort(p);adjusted=np.empty(len(p))
    adjusted[order]=np.minimum(1,np.maximum.accumulate((len(p)-np.arange(len(p)))*p[order]));return adjusted
def compare_groups(frame,group,cols,a,b):
    rows=[]
    for col in cols:
        x=frame.loc[frame[group].eq(a),col].dropna().to_numpy();y=frame.loc[frame[group].eq(b),col].dropna().to_numpy()
        t,p=stats.ttest_ind(x,y,equal_var=False)
        diff=x.mean()-y.mean();vx=x.var(ddof=1)/len(x);vy=y.var(ddof=1)/len(y);se=np.sqrt(vx+vy)
        dof=(vx+vy)**2/(vx*vx/(len(x)-1)+vy*vy/(len(y)-1));margin=stats.t.ppf(.975,dof)*se
        pooled=np.sqrt(((len(x)-1)*x.var(ddof=1)+(len(y)-1)*y.var(ddof=1))/(len(x)+len(y)-2))
        rows.append([col,len(x),len(y),x.mean(),y.mean(),diff,t,p,diff-margin,diff+margin,diff/pooled])
    r=pd.DataFrame(rows,columns=['показатель','n_A','n_B','mean_A','mean_B','A-B','t','p','CI_low','CI_high','Cohen_d'])
    r['значимо без поправки']=r.p<.05;r['p_Holm']=holm(r.p);r['вывод_0.05']=np.where(r.p_Holm<.05,'H0 отклонена','H0 не отклонена');return r
birth_tests=compare_groups(birth,'pol(1-м, 0-д)',['rost','ves','ИМТ'],1,0);display(birth_tests)
birth_tests.to_csv('birth_welch.csv',index=False)

Сравнение мальчики минус девочки:

**rost**: разность 0.674, 95% ДИ [0.424; 0.924], p=1.442e-07, p Holm=2.883e-07; есть статистические основания отвергнуть равенство средних.

**ves**: разность 136.915, 95% ДИ [89.600; 184.229], p=1.765e-08, p Holm=5.294e-08; есть статистические основания отвергнуть равенство средних.

**ИМТ**: разность 0.169, 95% ДИ [0.046; 0.292], p=0.006948, p Holm=0.006948; есть статистические основания отвергнуть равенство средних.

Группы независимы, дисперсии не приравниваются. Большой объём смягчает ненормальность для среднего, но независимость наблюдений и отсутствие семейной кластеризации требуют проверки по дизайну сбора. Значимость не равна клинической значимости.

In [ ]:
display(Markdown('## Итоговый вывод'))
print(f'Всего {len(birth)} новорождённых: {(birth["pol(1-м, 0-д)"] == 1).sum()} мальчиков '
      f'и {(birth["pol(1-м, 0-д)"] == 0).sum()} девочек.')
print('H0 о нормальности отклонена для:', ', '.join(normal.loc[normal.p < .05, 'показатель']))
for row in birth_tests.to_dict('records'):
    print(f"{row['показатель']}: средние {row['mean_A']:.3f} и {row['mean_B']:.3f}; "
          f"t={row['t']:.4f}, p={row['p']:.6g}, p Holm={row['p_Holm']:.6g}; {row['вывод_0.05']}.")
